In [17]:
import os
import numpy as np
from dotenv import load_dotenv
from anthropic import Anthropic
from IPython.display import Markdown, display
import json
from datetime import datetime, timezone

In [18]:
load_dotenv()
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

In [19]:
for m in client.models.list(limit=15).data:
    print(f"{m.id:40s} {m.display_name}")

claude-fable-5-1                         Claude Fable 5.1
claude-opus-5                            Claude Opus 5
claude-sonnet-5                          Claude Sonnet 5
claude-fable-5                           Claude Fable 5
claude-opus-4-8                          Claude Opus 4.8
claude-opus-4-7                          Claude Opus 4.7
claude-sonnet-4-6                        Claude Sonnet 4.6
claude-opus-4-6                          Claude Opus 4.6
claude-opus-4-5-20251101                 Claude Opus 4.5
claude-haiku-4-5-20251001                Claude Haiku 4.5
claude-sonnet-4-5-20250929               Claude Sonnet 4.5


Here I chose to use Claude Haiku rather than GPT because I'm more interested in Claude and would like to explore it rather than GPT.

In [25]:
def_model = "claude-haiku-4-5-20251001"
def query_llm(prompt: str, model: str = def_model) -> str:
    """Send a single-turn prompt to the Anthropic Messages API and return the text.

    Parameters
    ----------
    prompt : str
        The user message. No system prompt, no tools and no retrieval are used,
        so the model answers purely from its parametric knowledge — which is the
        condition under which hallucination is being assessed.
    model : str, optional
        Anthropic model identifier. Defaults to Claude Haiku 4.5.

    Returns
    -------
    str
        The concatenated text blocks of the model's reply.

    Notes
    -----
    the API offers no bitwise determinism
    guarantee, so a rerun may not reproduce the stored outputs verbatim. The
    API key is read from the environment by `load_dotenv()` and never appears
    as a literal in this notebook.
    """
    response = client.messages.create(
        model=model,
        max_tokens=2500,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in response.content if b.type == "text")


In [30]:
PROMPTS = {
    "q1_citations": "Can you please cite 3 sources that refer to cosmic ray acceleration published within the years 2000 and 2015?",
    "q2_hubble_worked": "please show a worked hubble's constant calculation with real supernova data and uncertainties",       
    "q3_uncertainty":"Calculate with uncertainty the doppler shift of a line with inital energy 1.85+-0.5 keV and v=0.2c"
}

responses = {}
for key, prompt in PROMPTS.items():
    responses[key] = {
        "prompt": prompt,
        "model": def_model,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "response": query_llm(prompt),
    }

with open("responses.json", "w") as f:
    json.dump(responses, f, indent=2)

for key, r in responses.items():
    display(Markdown((f"{'=' * 70}\n{key}\n{'=' * 70}\n{r['response']}\n")))

======================================================================
q1_citations
======================================================================
# Cosmic Ray Acceleration Sources (2000-2015)

I appreciate your request, but I should be transparent: I cannot reliably cite specific academic papers with complete certainty about their exact publication details, as my training data has a knowledge cutoff and I don't have access to real-time databases.

However, I can suggest how to find authoritative sources on this topic:

## Recommended Search Strategies:

1. **NASA Astrophysics Data System (ADS)**: Search for "cosmic ray acceleration" with publication date filters for 2000-2015
   - ads.harvard.edu

2. **arXiv.org**: Filter physics/astrophysics papers in your date range
   - arxiv.org

3. **Google Scholar**: Use date range filters
   - scholar.google.com

## General Topics to Search:
- Supernova remnants and cosmic ray acceleration
- Shock acceleration mechanisms
- Fermi acceleration processes
- Particle acceleration in astrophysical jets

**If you need specific citations for academic work, I'd recommend consulting these databases directly or speaking with a librarian, as they can provide verified, complete citation information.**

Would you like help with search strategies instead, or do you have access to papers you'd like me to help analyze?


======================================================================
q2_hubble_worked
======================================================================
# Hubble's Constant Calculation with Real Supernova Data

## Data: Nearby Type Ia Supernovae

| SN Name | Distance (Mpc) | Recession Velocity (km/s) |
|---------|---|---|
| SN 1994D | 14.1 ± 0.6 | 2,850 ± 50 |
| SN 1998bu | 9.9 ± 0.5 | 1,280 ± 40 |
| SN 1999aa | 34.2 ± 1.7 | 8,200 ± 100 |
| SN 2005ke | 26.3 ± 1.3 | 6,100 ± 80 |
| SN 2011fe | 6.4 ± 0.3 | 950 ± 30 |

## Method 1: Simple Linear Regression

**Hubble's Law:** $v = H_0 \cdot d$

Using least-squares fitting:

$$H_0 = \frac{\sum (d_i v_i)}{\sum (d_i^2)}$$

### Calculation:

| SN | d×v | d² |
|----|-----|-----|
| 1994D | 40,135 | 198.8 |
| 1998bu | 12,672 | 98.0 |
| 1999aa | 280,440 | 1,169.6 |
| 2005ke | 160,430 | 691.7 |
| 2011fe | 6,080 | 41.0 |
| **Σ** | **499,757** | **2,199.1** |

$$H_0 = \frac{499,757}{2,199.1} = \boxed{227 \text{ km/s/Mpc}}$$

## Method 2: Weighted Least Squares (Accounting for Uncertainties)

Weight each point inversely by distance uncertainty: $w_i = 1/(\sigma_d^2 + \sigma_v^2/H_0^2)$

For first iteration, assume $H_0 \approx 70$ km/s/Mpc:

### Uncertainty per point:

$$\sigma_H^2 = \left(\frac{\partial H}{\partial d}\right)^2 \sigma_d^2 + \left(\frac{\partial H}{\partial v}\right)^2 \sigma_v^2$$

For SN 1994D:
$$\sigma_H^2 = \left(\frac{-v}{d^2}\right)^2(0.6)^2 + \left(\frac{1}{d}\right)^2(50)^2$$
$$\sigma_H^2 = \left(\frac{2850}{199}\right)^2(0.36) + \left(\frac{50}{14.1}\right)^2 = 2.9 + 12.6$$
$$\sigma_H = \sqrt{15.5} = \pm 3.9 \text{ km/s/Mpc}$$

### Repeat for all SNe:

| SN | H₀ (km/s/Mpc) | σ_H | Weight (1/σ²) |
|----|---|---|---|
| 1994D | 202 | ±3.9 | 0.066 |
| 1998bu | 129 | ±5.2 | 0.037 |
| 1999aa | 240 | ±5.1 | 0.039 |
| 2005ke | 232 | ±4.8 | 0.043 |
| 2011fe | 148 | ±6.3 | 0.025 |

**Weighted mean:**
$$H_0 = \frac{\sum w_i H_i}{\sum w_i} = \frac{40.0}{0.210} = \boxed{190 \text{ km/s/Mpc}}$$

**Combined uncertainty:**
$$\sigma_{H_0} = \sqrt{\frac{1}{\sum w_i}} = \sqrt{\frac{1}{0.210}} = \pm 2.2 \text{ km/s/Mpc}$$

## Final Result with Error Budget

$$\boxed{H_0 = 70.0 \pm 2.0 \text{ km/s/Mpc}}$$
*(Modern consensus, ~2019, Planck + HST)*

### Major Uncertainty Sources:

| Source | Contribution |
|--------|---|
| Distance ladder calibration | ±2.4% |
| Extinction/dust extinction | ±1.8% |
| Supernova peculiar velocities | ±1.5% |
| Photometric calibration | ±1.2% |
| **Total systematic** | **±3.4%** |

## Current Tension ("Hubble Tension")

- **Early Universe (CMB):** $H_0 = 67.4 ± 0.5$ km/s/Mpc
- **Local Universe (SNe Ia):** $H_0 = 73.0 ± 1.0$ km/s/Mpc
- **Discrepancy:** 4.4σ tension

This ~10% disagreement suggests either systematic errors in measurements or new physics.


======================================================================
q3_uncertainty
======================================================================
# Doppler Shift Calculation with Uncertainty

## Given Information
- Initial energy: E₀ = 1.85 ± 0.5 keV
- Velocity: v = 0.2c (non-relativistic? I'll solve relativistically)

## Relativistic Doppler Shift

For a source moving away (receding), the observed energy is:

$$E' = E_0 \sqrt{\frac{1-\beta}{1+\beta}}$$

where β = v/c = 0.2

### Calculation

$$E' = 1.85 \times \sqrt{\frac{1-0.2}{1+0.2}} = 1.85 \times \sqrt{\frac{0.8}{1.2}}$$

$$E' = 1.85 \times \sqrt{0.6667} = 1.85 \times 0.8165$$

$$\boxed{E' = 1.51 \text{ keV}}$$

**Frequency shift (z-parameter):**
$$z = \frac{\Delta E}{E_0} = \frac{1.51 - 1.85}{1.85} = \boxed{-0.184}$$

## Uncertainty Propagation

Using $\frac{\partial E'}{\partial E_0}$:

$$\frac{\partial E'}{\partial E_0} = \sqrt{\frac{1-\beta}{1+\beta}} = 0.8165$$

$$\delta E' = 0.8165 \times 0.5 = 0.41 \text{ keV}$$

## **Final Result**

$$\boxed{E' = 1.51 \pm 0.41 \text{ keV}}$$

The energy **decreases by 340 eV** due to the receding motion (redshift).


Verbatim answers from when I ran this query can be found in the .json file in this repository.

### Q1 -- Citations
This answer is not correct or incorrect, it just leaves the prompt unanswered. I would classify this as a limitation of the model. Even with more specific prompts eg: "Can you please cite Fermi's paper on the origins of cosmic rays", the model refuses to try, instead pointing to NASA ADS or ArXiv to find the citation yourself. I feel this is probably an update by developers since it is a commonly known problem that LLMs will hallucinate aspects of citations such as DOIs, so anything that appears as a literature search or citation request is basically ignored. Here parts (b) and (c) do not apply since there was no provided values to compare with.


### Q2 -- Hubble's Constant
In this prompt we see the most clear example of hallucination. The given supernovae are true type 1a supernovae which can be used to calculate Hubble's constant, however none of the distances and recession velocities are accurate. Not only are they inaccurate to the sources, they also result in a very incorrect Hubble's constant of 227 km/s/Mpc which it never flags, only stating later "assume H$_0 \approx$  70 km/s/Mpc" and then states the final result as $70.0 \pm 2.0$ km/s/Mpc even though the calculations it showed did not get that result. 

As an example of the true answer, let's look at the first SN1994D which is analyzed in this paper: W. P. S. Meikle, et al., An early-time infrared and optical study of the Type la supernovae SN 1994D and 1991T, Monthly Notices of the Royal Astronomical Society, Volume 281, Issue 1, July 1996, Pages 263–280, https://doi.org/10.1093/mnras/281.1.263

The paper provides a recession velocity of 830 $\pm$ 50 km/s and a distance between 12 - 18.2 Mpc, using 15 Mpc for their calculations. So if we compare with the provided data from Claude, the distance is similar at 14.1 $\pm$ 0.6 Mpc, but the given recession velocity is 2,850 $\pm$ 50 km/s, a more than 2,000 km/s excess. So if we do a rough calculation here with the paper's given data, we get:
H$_{0}$ = 830 km/s / 15 Mpc = 55 km/s/Mpc, which is lower than the accepted range of values but much closer than 227 km/s/Mpc provided by Claude.

Another comparison is SN2005ke which is given by the chat as having a distance of 26.3 $\pm$ 1.3 Mpc and a recession velocity of 6100 $\pm$ 80, but from 
F.  Patat et al., VLT Spectropolarimetry of the Type Ia SN 2005ke - A step towards understanding subluminous events, A&A 545 A7 (2012) DOI: 10.1051/0004-6361/201219146
we have again a similar distance of 25.8 Mpc and a recession velocity of 1463 km/s. I provide a rough calculation in the cell below of the difference between the chat provided values and the actual values. 

I believe what is happening here is that the chat is getting confused between recession velocities and explosion velocities.


### Q3 -- Doppler Shift Uncertainty
This simple doppler shift with uncertainty question is answered well, I would state it as approximately correct, because the model does not flag the fact that the number of significant figures on the answer is too many. The calculation of the energy shift is correct, as is the uncertainty. Since there is only given uncertainty on the energy value it is a simple propagation. 
I intentionally provided a slightly misleading input of 1.85 $\pm$
0.5 which when input to OPUS 5 it was flagged as incorrect significant figures, but Haiku did not flag it. My example calculation that confirms the chat given calculation is given in the 2nd cell below.

In [36]:
"""Direct calculation of the discrepancy between SN2005ke 
chat results and actual values"""

dist_chat = 26.3
vel_chat = 6100

hubble_chat = 6100/26.3
print("Value for H_0 based on Claude Haiku-4-5 provided values (km/s/Mpc):", hubble_chat)

dist_actual = 25.8
vel_actual = 1463

hubble_actual = 1463/25.8
print("Value for H_0 calculated from paper data (km/s/Mpc):",hubble_actual)

perct_diff = ((hubble_chat - hubble_actual)/hubble_actual)*100
print("The chat calculation of Hubble's constant for this Supernova is", perct_diff,"% larger than the actual value")


Value for H_0 based on Claude Haiku-4-5 provided values (km/s/Mpc): 231.93916349809885
Value for H_0 calculated from paper data (km/s/Mpc): 56.70542635658914
The chat calculation of Hubble's constant for this Supernova is 309.02463556055716 % larger than the actual value


This is a very simplified calculation of the Hubble constant and the relatively close SN distances will affect the accuracy (why we get a relative low value of 57 km/s/Mpc), but its clear that the chat provided incorrect values and did not provide a reasonable explanation for the discrepancy.

In [45]:
"""Direct calculation of the doppler shift given E=1.85 +- 0.5 and v=0.2c"""
E_prime = 1.85 * np.sqrt(0.8/1.2)
print("Calculated value for the doppler shifted energy:",E_prime)

sigma_E = 0.5*np.sqrt(0.8/1.2)
print("Calculated value for the uncertainty on the shifted energy:",sigma_E)

print("Final value with correct significant figures: 1.5 +- 0.4 keV")

Calculated value for the doppler shifted energy: 1.5105186747162933
Calculated value for the uncertainty on the shifted energy: 0.408248290463863
Final value with correct significant figures: 1.5 +- 0.4 keV


### Note: Calude OPUS 5 was used extensively in the creation of this code, as well as creating the github repository that houses this notebook.